# 09 — Chain of Draft Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Chain of Draft: Iterative Refinement

In [3]:
topic = "Write a professional email asking for a meeting with a busy executive."
draft1 = chat([{"role":"user","content":f"Write a first draft for: {topic}"}],
              temperature=0.7, max_tokens=150)
print("DRAFT 1:")
print(draft1.strip())


DRAFT 1:
Subject: Request for Meeting to Discuss [Topic]

Dear [Executive's Name],

I hope this email finds you well. I am reaching out to request a meeting with you to discuss [Topic], which I believe has significant implications for [Company/Industry]. I have been following your work and achievements at [Company], and I am impressed by your leadership and vision.

I would greatly appreciate the opportunity to discuss [Topic] with you in more detail and explore ways in which we can work together to drive [Desired Outcome]. I am confident that a meeting with you would be a valuable use of your time, and I am prepared to come well-prepared and focused.

Would it be possible to schedule a meeting with you at your earliest convenience? I


In [4]:
critique = chat([
    {"role":"user","content":f"Write a first draft for: {topic}"},
    {"role":"assistant","content":draft1},
    {"role":"user","content":"Critique this draft. What are 3 specific weaknesses?"}
], temperature=0, max_tokens=150)
print("CRITIQUE:")
print(critique.strip())


CRITIQUE:
Here are three specific weaknesses in the draft:

1. **Lack of specificity**: The email mentions "significant implications for [Company/Industry]" but doesn't provide any concrete details about what those implications are or how they relate to the executive's work. This makes it difficult for the executive to understand the relevance and value of the meeting. To improve this, the email could provide more specific information about the topic and its connection to the executive's work.

2. **Overly generic language**: Phrases like "I am impressed by your leadership and vision" are generic and don't add much value to the email. They also come across as insincere or formulaic. To improve this, the email could use more specific and sincere language that


In [5]:
draft2 = chat([
    {"role":"user","content":f"Write a first draft for: {topic}"},
    {"role":"assistant","content":draft1},
    {"role":"user","content":"Critique this draft. What are 3 specific weaknesses?"},
    {"role":"assistant","content":critique},
    {"role":"user","content":"Now rewrite the email addressing all the weaknesses identified."}
], temperature=0.3, max_tokens=200)
print("REFINED DRAFT:")
print(draft2.strip())


REFINED DRAFT:
Subject: Request for Meeting to Discuss Strategic Opportunities in [Industry]

Dear [Executive's Name],

I hope this email finds you well. I am reaching out to request a meeting with you to discuss strategic opportunities in the [Industry] sector, where I believe [Company] has a unique position to drive innovation and growth. Specifically, I would like to explore how our company's [Technology/Product/Service] can be leveraged to address the industry's pressing challenges, such as [Challenge 1] and [Challenge 2].

I have been following your company's efforts to [Specific Initiative or Achievement] and am impressed by the impact it has had on the industry. I believe that a meeting with you would be a valuable opportunity to discuss how our companies can collaborate to drive meaningful change and create new opportunities for growth.

I have prepared a brief presentation outlining our company's approach and potential areas of collaboration. I would be happy to share this wit

## Chain of Draft for Code

In [6]:
task = "Write a Python function to find the second largest number in a list."
d1 = chat([{"role":"user","content":task}], temperature=0, max_tokens=150)
print("Draft 1:")
print(d1.strip())


Draft 1:
**Finding the Second Largest Number in a List**

Here's a Python function that finds the second largest number in a list:

```python
def find_second_largest(numbers):
    """
    Find the second largest number in a list.

    Args:
        numbers (list): A list of numbers.

    Returns:
        int or float: The second largest number in the list. If the list has less than two unique numbers, returns None.

    Raises:
        ValueError: If the input list is empty.
    """
    if not numbers:
        raise ValueError("Input list is empty")

    # Remove duplicates and sort the list in descending order
    unique_numbers = sorted(set(numbers), reverse=True)

    # Check if the list


In [7]:
c1 = chat([
    {"role":"user","content":task},
    {"role":"assistant","content":d1},
    {"role":"user","content":"What edge cases does this miss? List them."}
], temperature=0, max_tokens=100)
print("Issues found:")
print(c1.strip())


Issues found:
Here are some edge cases that the `find_second_largest` function might miss:

1. **Empty list**: The function raises a `ValueError` when the input list is empty, which is a good start. However, it might be more Pythonic to return `None` or a specific value to indicate that the list is empty.

2. **List with less than two unique numbers**: The function returns `None` when the list has less than two unique numbers, which is correct.


In [8]:
d2 = chat([
    {"role":"user","content":task},
    {"role":"assistant","content":d1},
    {"role":"user","content":"What edge cases does this miss?"},
    {"role":"assistant","content":c1},
    {"role":"user","content":"Rewrite the function handling all edge cases. Add docstring."}
], temperature=0, max_tokens=250)
print("Final version:")
print(d2.strip())


Final version:
**Finding the Second Largest Number in a List**

Here's a revised version of the `find_second_largest` function that handles all edge cases:

```python
def find_second_largest(numbers):
    """
    Find the second largest number in a list.

    Args:
        numbers (list): A list of numbers.

    Returns:
        int or float: The second largest number in the list. If the list has less than two unique numbers, returns None. If the list is empty, returns None.

    Raises:
        TypeError: If the input is not a list.
    """
    if not isinstance(numbers, list):
        raise TypeError("Input must be a list")

    if len(numbers) < 2:
        return None

    # Remove duplicates and sort the list in descending order
    unique_numbers = sorted(set(numbers), reverse=True)

    # Check if the list has at least two unique numbers
    if len(unique_numbers) < 2:
        return None

    # Return the second largest number
    return unique_numbers[1]
```

This revised funct

## Compressed Chain of Draft (Single Prompt)

In [9]:
cod_prompt = (
    "Task: Write a 3-sentence product description for a wireless noise-canceling headphone.\n\n"
    "Step 1 - Draft: Write a quick first attempt.\n"
    "Step 2 - Critique: Note 2 weaknesses in that draft.\n"
    "Step 3 - Final: Rewrite addressing the weaknesses.\n\n"
    "Output all three steps."
)
result = chat([{"role":"user","content":cod_prompt}], temperature=0.5, max_tokens=300)
print(result.strip())


**Step 1 - Draft:**

Our wireless noise-canceling headphones offer an immersive listening experience with crystal-clear sound and advanced noise-cancellation technology. With up to 20 hours of battery life, you can enjoy uninterrupted music, podcasts, or phone calls on the go. The sleek design and lightweight build ensure a comfortable fit, making it perfect for long flights, commutes, or daily use.

**Step 2 - Critique:**

Two weaknesses in this draft are:

1. The description is too generic and doesn't stand out from other similar products. It would be better to highlight specific features or benefits that set this product apart from the competition.
2. The language is somewhat vague, with phrases like "crystal-clear sound" and "immersive listening experience." Using more specific and technical terms would help to convey a sense of quality and expertise.

**Step 3 - Final:**

Our wireless noise-canceling headphones feature proprietary Active Noise Cancellation (ANC) technology, which 